### New crawler

In [1]:
import requests
import pandas as pd
import time
import random
import re
import json
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

class CourseraUnifiedScraper:
    def __init__(self):
        self.base_api_url = "https://api.coursera.org/api/courses.v1"
        self.filename = "coursera_full_data_2026.csv"
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        }
        self.session = requests.Session()
        
        # Human-readable mapping
        self.type_map = {
            "v2.ondemand": "Individual Course",
            "v2.capstone": "Capstone Project",
            "v2.specialization": "Specialization",
            "v2.professional-certificate": "Professional Certificate"
        }

    def fetch_course_index(self, limit=None):
        target_str = f"{limit} courses" if limit else "the ENTIRE database"
        print(f"🚀 [Step 1/2] Fetching {target_str} and extracting Skills...")
        
        all_courses = []
        start = 0
        batch_size = 100 
        # Fields include domainTypes for Skills and difficultyLevel for accuracy
        fields = "name,slug,workload,primaryLanguages,difficultyLevel,courseType,partnerIds,domainTypes"
        includes = "partnerIds"

        while True:
            if limit and len(all_courses) >= limit:
                break

            params = {
                "start": start,
                "limit": min(batch_size, limit - len(all_courses)) if limit else batch_size,
                "fields": fields,
                "includes": includes
            }
            
            try:
                response = self.session.get(self.base_api_url, params=params, timeout=15)
                if response.status_code != 200: break
                
                data = response.json()
                elements = data.get('elements', [])
                if not elements: break

                # Map Organizations
                linked = data.get('linked', {})
                partners_map = {p['id']: p['name'] for p in linked.get('partners.v1', [])}

                for item in elements:
                    # 1. Parse Skills (Domain Types)
                    domains = item.get("domainTypes", [])
                    skill_tags = [d.get('subdomainId', '').replace('-', ' ').title() for d in domains if d.get('subdomainId')]
                    
                    # 2. Map Course Type & Payment Model
                    raw_type = item.get("courseType", "Course")
                    if "degree" in item.get('name', '').lower() or raw_type == "v2.degree":
                        payment_status = "Paid (Degree)"
                    elif raw_type in ["v2.specialization", "v2.professional-certificate", "v2.capstone"]:
                        payment_status = "Subscription"
                    else:
                        payment_status = "Free Audit / Pay for Cert"

                    all_courses.append({
                        "Course title": item.get("name"),
                        "Course_Type": self.type_map.get(raw_type, raw_type),
                        "Payment_Model": payment_status,
                        "Organization": partners_map.get(item.get('partnerIds', [None])[0], "Coursera"),
                        "Skills": ", ".join(skill_tags),
                        "Workload": item.get("workload", "N/A"),
                        "Difficulty_level": item.get("difficultyLevel", "N/A"),
                        "Language": ",".join(item.get("primaryLanguages", [])),
                        "Course_link": f"https://www.coursera.org/learn/{item.get('slug')}",
                        "Ratings": "N/A",
                        "Review count": 0
                    })
                
                start += batch_size
                print(f"   Collected {len(all_courses)} course skeletons...", end='\r')
                time.sleep(0.3)
            except Exception as e:
                print(f"\n❌ API Error: {e}")
                break
        
        return all_courses

    def scrape_detailed_stats(self, course_data):
        url = course_data["Course_link"]
        try:
            time.sleep(random.uniform(0.5, 1.2))
            response = self.session.get(url, timeout=15)
            if response.status_code == 200:
                content = response.text
                
                # Regex for Ratings/Reviews
                r_match = re.search(r'"ratingValue"\s*:\s*([\d\.]+)', content)
                c_match = re.search(r'"reviewCount"\s*:\s*(\d+)', content)
                if r_match: course_data["Ratings"] = round(float(r_match.group(1)), 1)
                if c_match: course_data["Review count"] = int(c_match.group(1))

                # Regex for Difficulty fix if API failed
                if course_data["Difficulty_level"] == "N/A":
                    d_match = re.search(r'"difficultyLevel"\s*:\s*"([^"]+)"', content)
                    if d_match: 
                        course_data["Difficulty_level"] = d_match.group(1).capitalize()
                    else:
                        for level in ["Beginner", "Intermediate", "Advanced", "Mixed"]:
                            if re.search(rf'{level}\s*Level', content, re.IGNORECASE):
                                course_data["Difficulty_level"] = level
                                break
        except Exception: pass
        return course_data

    def run(self, limit=None):
        # Step 1
        courses = self.fetch_course_index(limit=limit)
        
        # Step 2
        print("\n🚀 [Step 2/2] Enriching with Ratings, Review Counts, and Difficulty Fixes...")
        enriched_data = []
        with ThreadPoolExecutor(max_workers=5) as executor:
            future_to_course = {executor.submit(self.scrape_detailed_stats, c): c for c in courses}
            processed = 0
            total = len(courses)
            for future in as_completed(future_to_course):
                enriched_data.append(future.result())
                processed += 1
                if processed % 50 == 0:
                    print(f"   Progress: {processed}/{total} processed...", end='\r')

        # Formatting Output
        df = pd.DataFrame(enriched_data)
        
        # Exact column order you requested
        cols = [
            "Course title", "Course_Type", "Organization", "Skills", 
            "Workload", "Difficulty_level", "Language", "Course_link", 
            "Ratings", "Review count", "Payment_Model"
        ]
        df = df[[c for c in cols if c in df.columns]]
        
        # Summary & Save
        print("\n" + "="*50)
        print("📊 COLLECTION COMPLETE")
        print(f"Total records: {len(df)}")
        print("-" * 30)
        print("Payment Models found:")
        print(df["Payment_Model"].value_counts())
        print("="*50)
        
        df.to_csv(self.filename, index=False, encoding='utf_8_sig')
        print(f"✅ Data saved to {self.filename}")

if __name__ == "__main__":
    scraper = CourseraUnifiedScraper()
    # Set limit=None for full crawl, or limit=500 for a test run
    scraper.run(limit=None)


🚀 [Step 1/2] Fetching the ENTIRE database and extracting Skills...
   Collected 18910 course skeletons...
🚀 [Step 2/2] Enriching with Ratings, Review Counts, and Difficulty Fixes...
   Progress: 18900/18910 processed...
📊 COLLECTION COMPLETE
Total records: 18910
------------------------------
Payment Models found:
Payment_Model
Free Audit / Pay for Cert    18789
Subscription                   120
Paid (Degree)                    1
Name: count, dtype: int64
✅ Data saved to coursera_full_data_2026.csv
